# 양서연 · TOPIS 링크 기본표와 도로 통행시간표
팀 계획에 맞춰 44개월 TOPIS와 시간별 날씨를 결합한다. 원본은 수정하지 않는다.

1. 44개월 모두 존재하는 링크만 선정한다.
2. 링크ID×날짜×시간 기본표에는 연구기간과 앞 4주 buffer, 전 시간대를 보존한다.
3. 도로×방향 표는 연구기간의 평일 출퇴근 시간만 사용하고 한국 공휴일을 제외한다.
4. 시스템 공백은 행을 유지하고 NaN 및 결측 사유로 기록한다.

## 1. 환경·경로 확인 — 저장 없음

In [ ]:
import importlib.util, os, sys
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'README.md').is_file() and (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / 'data' / 'raw'  # 원본 위치 (io_utils.data_root, 드라이브에서 받아 푼 곳)
LOG_DIR = PROJECT_ROOT / 'logs' / 'ysy' / 'topis_weather_merge'
OUT_DIR = PROJECT_ROOT / 'data' / 'derived' / 'topis_weather'
RUN_BUILD = False  # True로 바꿀 때만 표를 새로 만든다 (기본은 이미 만든 결과를 읽기만)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Python:', sys.executable)
print('프로젝트:', PROJECT_ROOT)
print('데이터:', DATA_ROOT)
print('새로 만들기:', RUN_BUILD)

## 2. 패키지·입력 확인 — 저장 없음

In [ ]:
required = ['pandas', 'pyarrow', 'holidays', 'yaml']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'필요 패키지가 없습니다: {missing}')
topis_paths = [p for p in sorted((DATA_ROOT / 'topis_speed').glob('20??-??.csv'))
               if p.stem[-2:] != '10' and p.stem[-7:] not in {'2022-07', '2022-08'}]  # Release 50개월 중 결합용 44개월
weather_paths = sorted((PROJECT_ROOT / 'data' / 'shared').rglob('weather_hourly_*.csv'))
print('TOPIS 월 파일:', len(topis_paths))
print('시간별 날씨 파일:', len(weather_paths))
if len(topis_paths) != 44 or len(weather_paths) != 2:
    raise RuntimeError('입력 파일 수를 확인하세요.')


## 3. 공통 기준 확인 — 저장 없음

In [ ]:
from src.io_utils import HOUR_COLS, read_topis_gaps
from src.join.topis_weather_merge import load_periods, load_roads, run_merge
periods, commute_hours = load_periods(PROJECT_ROOT)
roads = load_roads(PROJECT_ROOT)
gaps = read_topis_gaps(PROJECT_ROOT)
print('시간 열:', HOUR_COLS[0], '~', HOUR_COLS[-1])
print('출퇴근 종료시각:', commute_hours)
print('연구기간:', len(periods))
print('분석 도로:', len(roads))
print('TOPIS 시스템 공백:', len(gaps))


## 4. 전체 결합 — **`RUN_BUILD = True`일 때만 대용량 읽기와 파일 저장**
`RUN_BUILD = True`이면 링크 기본표와 도로 통행시간표를 `data/derived/topis_weather/`에 Parquet으로 새 버전 번호를 붙여 저장한다 (지금 있는 v04 · v05 다음 번호). 프로젝트에는 소량 샘플 · 월별 요약 · 공통 링크 목록 · 실행 로그만 저장한다. 기존 파일은 덮어쓰지 않는다.
기본값 `False`에서는 아무것도 만들지 않고, 이미 만든 결과 파일의 행 수만 보여 준다.

In [ ]:
if not RUN_BUILD:
    print('건너뜀: RUN_BUILD = False (표를 새로 만들지 않음)')
else:
    merge_log = run_merge(DATA_ROOT, PROJECT_ROOT)
    for key, value in merge_log.items():
        print(f'{key}: {value}')

# 결과 확인: 이미 만든 표 (행 수는 Parquet 머리말만 읽어서 빠르다)
import pyarrow.parquet as pq
for path in sorted(OUT_DIR.glob('*.parquet')):
    print(f'{path.name}: {pq.ParquetFile(path).metadata.num_rows:,}행 · {path.stat().st_size / 1e6:,.1f}MB')

## 5. v03 → v04 재현 — **`RUN_BUILD = True`일 때만 Parquet 읽기와 파일 저장**
v03의 `observed_link_count < link_count` 행은 관측 링크만 합산되어 통행시간이 과소 추정된다. 해당 행의 `travel_time_seconds`와 `road_speed_kmh`를 `NaN`으로 바꾸고 기존 파일을 덮어쓰지 않은 채 v04로 저장한다. 샘플은 규칙 확인을 위해 정상 관측 100행과 결측 처리 사례 100행을 의도적으로 함께 저장하므로 전체 결측률을 대표하지 않는다. 출력 파일이 이미 있으면 중단한다.
v04는 이미 만들어 두었고 v03은 정리하면서 지웠다. 그래서 이 셀은 v04를 만든 방법의 기록이고, 기본값에서는 v04 변환 로그만 보여 준다.

In [ ]:
import json
from src.join.topis_weather_merge import convert_v03_to_v04

run_date = '20261122'
v04_log_path = LOG_DIR / f'topis_weather_merge_{run_date}_v04.json'
if not RUN_BUILD:
    print('건너뜀: RUN_BUILD = False (v04는 이미 있음)')
else:
    convert_v03_to_v04(
        source_path=OUT_DIR / f'commute_{run_date}_v03.parquet',
        output_path=OUT_DIR / f'commute_{run_date}_v04.parquet',
        sample_path=PROJECT_ROOT / 'data' / 'sample' / f'topis_weather_commute_sample_{run_date}_v04.csv',
        log_path=v04_log_path,
    )

# 결과 확인: v04 변환 로그
v04_log = json.loads(v04_log_path.read_text(encoding='utf-8'))
for key in ['run_at', 'method_version', 'output_rows', 'v03_travel_time_nan_rows', 'v04_travel_time_nan_rows', 'v04_nan_ratio', 'wording']:
    print(f'{key}: {v04_log.get(key)}')

## 6. Past run history (read-only)
Reads both the new personal log folder and the former shared log location. This cell does not run the merge.

In [ ]:
log_roots = [LOG_DIR, PROJECT_ROOT / 'logs']
history_files = []
for root in log_roots:
    if root.is_dir():
        history_files.extend([*root.glob('topis_weather_*.json'), *root.glob('traffic_october_*.json')])
history_files = sorted(set(path.resolve() for path in history_files))
print('Past run records:', len(history_files))
for path in history_files:
    print(path.relative_to(PROJECT_ROOT))


## 7. 제한사항
겨울 평소값 기간의 10월 30~31일 TOPIS 원자료가 Release에 없어 각 겨울 2일이 비어 있다. 여름 휴가 절정 주간은 팀에서 날짜를 확정하기 전까지 NA로 둔다. 두 사항을 임의 추정하지 않는다.